# GPT-5.4-mini(API)를 활용한 뉴스 프레이밍 분석

참고문헌:

Semetko, H. A., & Valkenburg, P. M. (2000). Framing European politics: A content analysis of press and television news. *Journal of communication*, 50(2), 93–109.

오나영·노지현·안순태 (2025). 세대 주거문제에서 사회적 재난으로 : 전세사기, 미디어 담론과 댓글. <한국언론학보>, 69권 5호, 5–42.

---

## 실행 환경 요약

- 모델: `gpt-5.4-mini` / API: OpenAI **Responses API** (`client.responses.create`) + JSON Schema 구조화 출력(`strict: true`)
- `reasoning_effort="none"` (분류·추출·요약 과제에 적합). `temperature`는 `reasoning_effort="none"`일 때만 지정할 수 있습니다.
- 이전 버전: [GPT-4.1 버전 저장소](https://github.com/HanJunLeeKR/News_Content_Analysis_with_GPT-4.1) (Chat Completions + `json_object`)
- 두 버전의 차이는 저장소 `README.md`의 「GPT-4.1 버전과의 차이」를 참고하세요.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 패키지 설치 (Colab/Jupyter용). 로컬 환경에서는 저장소의 requirements.txt 사용:
#   pip install -r requirements.txt
# ═══════════════════════════════════════════════════════════════════════
%pip install --upgrade openai pandas tqdm nest-asyncio


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 범용 뉴스 프레임 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════
import pandas as pd
from openai import AsyncOpenAI
import asyncio
import json
import os
import logging
import time
from typing import Optional, Tuple, Dict
import re
from tqdm import tqdm

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 1. 뉴스 프레임 정의 (Semetko & Valkenburg, 2000)
# ═══════════════════════════════════════════════════════════════════════
ALLOWED_FRAMES = {
    "갈등 프레임 (Conflict Frame)": {
        "description": "개인, 집단, 기관, 국가 간의 갈등, 대립, 불일치를 강조하는 프레임. 이견이나 논쟁을 중심으로 사건을 보도하며, 대립하는 양측의 입장을 부각시킴"
    },
    "책임귀인 프레임 (Attribution of Responsibility Frame)": {
        "description": "문제의 원인이나 해결책에 대한 책임을 특정 개인, 집단, 정부, 기관에 귀속시키는 프레임. 누가 문제를 야기했는지, 누가 해결해야 하는지를 명확히 함"
    },
    "인간적 흥미 프레임 (Human Interest Frame)": {
        "description": "사건을 인간적이고 감정적인 관점에서 접근하여 개인의 이야기, 감정, 일화를 강조하는 프레임. 독자의 감정적 반응을 유도하고 공감을 이끌어냄"
    },
    "경제적 결과 프레임 (Economic Consequences Frame)": {
        "description": "사건이나 문제가 개인, 집단, 지역, 국가에 미치는 경제적 영향이나 결과를 강조하는 프레임. 금전적 손실/이득, 비용, 경제적 파급효과를 중심으로 보도"
    },
    "도덕성 프레임 (Morality Frame)": {
        "description": "사건을 도덕적, 윤리적, 종교적 관점에서 해석하고 평가하는 프레임. 옳고 그름, 선과 악의 가치 판단을 내리며, 사회적 규범이나 윤리적 원칙을 강조"
    }
}

def format_allowed_frames():
    """프레임을 프롬프트용 문자열로 변환"""
    return "\n".join(
        f"- {name}: {info['description']}"
        for name, info in ALLOWED_FRAMES.items()
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 템플릿
# ═══════════════════════════════════════════════════════════════════════
PROMPT_TEMPLATE = """You are a media scholar. Classify the following Korean news article into exactly ONE of the five news frames listed below.

[NEWS ARTICLE]:
{text}

[FIVE NEWS FRAMES]:
{allowed_frames}

Instructions:
1. Identify the overall tone and emphasis of the article.
2. Select the single most prominent frame from the five options above.
3. Provide specific evidence from the article text.
4. Rate confidence as an integer between 1 and 100.

Output ONLY valid JSON. Do not add fields not in the schema."""

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 3. JSON 파서
# ═══════════════════════════════════════════════════════════════════════
class EnhancedJSONParser:
    """강화된 JSON 파싱 클래스"""

    @staticmethod
    def extract_json_robust(response_text: str) -> Dict:
        if not response_text or not response_text.strip():
            return EnhancedJSONParser._get_default_result("빈 응답")
        response_text = response_text.strip()

        try:
            if response_text.startswith('{') and response_text.endswith('}'):
                return json.loads(response_text)
        except json.JSONDecodeError:
            pass

        for pattern in [r'```json\s*\n?(.*?)\n?```', r'```\s*\n?(.*?)\n?```']:
            match = re.search(pattern, response_text, re.DOTALL | re.IGNORECASE)
            if match:
                try:
                    return json.loads(match.group(1).strip())
                except json.JSONDecodeError:
                    continue

        for pattern in [r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', r'\{(?:[^{}]|{[^{}]*})*\}']:
            for match in re.findall(pattern, response_text):
                try:
                    result = json.loads(match)
                    if EnhancedJSONParser._validate_result(result):
                        return result
                except json.JSONDecodeError:
                    continue

        extracted_data = {}
        for p in [r'"frame"\s*:\s*"([^"]+)"', r'frame["\']?\s*[:=]\s*["\']([^"\']+)["\']']:
            m = re.search(p, response_text, re.IGNORECASE)
            if m:
                extracted_data['frame'] = m.group(1).strip()
                break

        for p in [r'"reason"\s*:\s*"([^"]*(?:\\.[^"]*)*)"']:
            m = re.search(p, response_text, re.IGNORECASE | re.DOTALL)
            if m:
                extracted_data['reason'] = m.group(1).strip()
                break

        for p in [r'"confidence"\s*:\s*(\d+)']:
            m = re.search(p, response_text, re.IGNORECASE)
            if m:
                try:
                    extracted_data['confidence'] = int(m.group(1))
                except ValueError:
                    pass
                break

        if extracted_data and EnhancedJSONParser._validate_result(extracted_data):
            return extracted_data
        return EnhancedJSONParser._get_default_result("파싱 실패")

    @staticmethod
    def _validate_result(result: Dict) -> bool:
        return (
            isinstance(result, dict) and
            'frame' in result and
            isinstance(result['frame'], str) and
            result['frame'].strip() != ''
        )

    @staticmethod
    def _get_default_result(error_msg: str) -> Dict:
        return {'frame': '분류 실패', 'reason': f'파싱 오류: {error_msg}', 'confidence': 0}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 4. 뉴스 프레임 분석기 (GPT-5.4-mini 버전)
# ═══════════════════════════════════════════════════════════════════════
class NewsFrameAnalyzer:
    """뉴스 프레임 분석기 (GPT-5.4-mini용)"""

    def __init__(self, api_key, model, output_dir='.',
                 concurrent_requests=2, rate_limit_delay=1.0, batch_size=5,
                 reasoning_effort='none', text_verbosity='medium',
                 temperature: Optional[float] = None,
                 debug=False):
        # ── GPT-5.4-mini: temperature는 reasoning_effort='none'일 때만 유효 ──
        if temperature is not None and reasoning_effort != 'none':
            raise ValueError(
                f"temperature is only valid when reasoning_effort='none'. "
                f"Got reasoning_effort='{reasoning_effort}'."
            )
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.output_dir = output_dir
        self.concurrent_requests = concurrent_requests
        self.rate_limit_delay = rate_limit_delay
        self.batch_size = batch_size
        self.reasoning_effort = reasoning_effort
        self.text_verbosity = text_verbosity
        self.temperature = temperature
        self.debug = debug
        self.total_cost = 0.0
        self.semaphore = asyncio.Semaphore(concurrent_requests)
        self.parsing_successes = 0
        self.parsing_failures = 0
        logging.basicConfig(level=logging.WARNING, format='%(message)s')
        self.logger = logging.getLogger(__name__)
        os.makedirs(self.output_dir, exist_ok=True)

    def combine_title_text(self, title: str, text: str) -> str:
        if pd.isna(title): title = ""
        if pd.isna(text): text = ""
        title = str(title).strip()
        text = str(text).strip()
        combined = f"{title}\n\n{text}" if title and text else (title or text)
        return combined[:10000]

    async def analyze_single_article(self, title: str, text: str) -> Tuple[str, str, int, float]:
        async with self.semaphore:
            try:
                combined_text = self.combine_title_text(title, text)
                if not combined_text:
                    return "분류 실패", "빈 텍스트", 0, 0.0

                prompt = PROMPT_TEMPLATE.format(
                    text=combined_text,
                    allowed_frames=format_allowed_frames()
                )

                # ═══════════════════════════════════════════════════
                # GPT-5.4-mini API 호출
                # reasoning_effort options: none | low | medium | high | xhigh
                # temperature: only valid when reasoning_effort='none'
                # ═══════════════════════════════════════════════════
                api_params = dict(
                    model=self.model,
                    input=[{"role": "user", "content": prompt}],
                    max_output_tokens=1024,
                    reasoning={"effort": self.reasoning_effort},
                    text={
                        "verbosity": self.text_verbosity,
                        "format": {
                            "type": "json_schema",
                            "name": "news_frame_classification",
                            "strict": True,
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "frame": {
                                        "type": "string",
                                        "description": "The dominant news frame (one of the five Semetko & Valkenburg frames)"
                                    },
                                    "reason": {
                                        "type": "string",
                                        "description": "Specific evidence from the article supporting the frame classification"
                                    },
                                    "confidence": {
                                        "type": "integer",
                                        "description": "Confidence score (1-100)",
                                        "minimum": 1,
                                        "maximum": 100
                                    }
                                },
                                "required": ["frame", "reason", "confidence"],
                                "additionalProperties": False
                            }
                        }
                    }
                )
                if self.temperature is not None:
                    api_params["temperature"] = self.temperature

                response = await self.client.responses.create(**api_params)

                if hasattr(response, 'output_text') and response.output_text:
                    response_text = response.output_text
                else:
                    response_text = ""
                    for item in response.output:
                        if item.type == "message":
                            for content in item.content:
                                if hasattr(content, 'text'):
                                    response_text += content.text

                response_text = response_text.strip()
                if not response_text:
                    return "분류 실패", "빈 응답", 0, 0.0

                result = EnhancedJSONParser.extract_json_robust(response_text)
                frame      = result.get('frame', '분류 실패')
                reason     = result.get('reason', '파싱 실패')
                confidence = result.get('confidence', 0)
                try:
                    confidence = max(0, min(100, int(confidence)))
                except (ValueError, TypeError):
                    confidence = 0

                if frame != '분류 실패':
                    self.parsing_successes += 1
                else:
                    self.parsing_failures += 1

                input_tokens  = response.usage.input_tokens if hasattr(response.usage, 'input_tokens') else 0
                output_tokens = response.usage.output_tokens if hasattr(response.usage, 'output_tokens') else 0
                estimated_cost = (input_tokens * PRICE_INPUT_PER_1M / 1_000_000) + (output_tokens * PRICE_OUTPUT_PER_1M / 1_000_000)
                self.total_cost += estimated_cost

                return frame, reason, confidence, estimated_cost

            except Exception as e:
                if self.debug:
                    print(f"[DEBUG] Exception: {type(e).__name__}: {str(e)}")
                self.parsing_failures += 1
                return "분류 실패", f"오류: {str(e)[:50]}", 0, 0.0
            finally:
                await asyncio.sleep(self.rate_limit_delay)

    async def process_batch(self, batch_df) -> Tuple[list, list, list, list]:
        tasks = [
            self.analyze_single_article(row['title'], row['text'])
            for _, row in batch_df.iterrows()
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        frames, reasons, confidences, costs = [], [], [], []
        for result in results:
            if isinstance(result, Exception):
                frames.append("분류 실패")
                reasons.append(f"배치 오류: {str(result)[:50]}")
                confidences.append(0)
                costs.append(0.0)
                self.parsing_failures += 1
            else:
                frames.append(result[0])
                reasons.append(result[1])
                confidences.append(result[2])
                costs.append(result[3])
        return frames, reasons, confidences, costs

    async def analyze_dataframe(self, df, output_filename="news_frame_results.csv"):
        output_path = os.path.join(self.output_dir, output_filename)
        all_frames, all_reasons, all_confidences, all_costs = [], [], [], []
        start_time = time.time()
        pbar = tqdm(total=len(df), desc="뉴스 프레임 분석 중", unit="건")

        for i in range(0, len(df), self.batch_size):
            batch_df = df.iloc[i:min(i + self.batch_size, len(df))]
            try:
                frames, reasons, confidences, costs = await self.process_batch(batch_df)
                all_frames.extend(frames)
                all_reasons.extend(reasons)
                all_confidences.extend(confidences)
                all_costs.extend(costs)
                pbar.update(len(batch_df))

                batch_num = i // self.batch_size + 1
                if batch_num % 5 == 0:
                    n = len(df)
                    temp_df = df.copy()
                    temp_df['프레임'] = all_frames + [''] * (n - len(all_frames))
                    temp_df['근거'] = all_reasons + [''] * (n - len(all_reasons))
                    temp_df['신뢰도'] = all_confidences + [0] * (n - len(all_confidences))
                    temp_df['비용'] = all_costs + [0.0] * (n - len(all_costs))
                    temp_df.to_csv(output_path, encoding='utf-8-sig', index=False)
            except Exception as e:
                sz = len(batch_df)
                all_frames.extend(["분류 실패"] * sz)
                all_reasons.extend([f"배치 오류: {str(e)[:30]}"] * sz)
                all_confidences.extend([0] * sz)
                all_costs.extend([0.0] * sz)
                self.parsing_failures += sz
                pbar.update(sz)

        pbar.close()
        result_df = df.copy()
        result_df['프레임'] = all_frames
        result_df['근거'] = all_reasons
        result_df['신뢰도'] = all_confidences
        result_df['비용'] = all_costs
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        elapsed_time = time.time() - start_time
        self._print_completion_message(result_df, elapsed_time, output_path)
        return result_df

    def _print_completion_message(self, result_df, elapsed_time, output_path):
        total_count = len(result_df)
        frame_dist  = result_df['프레임'].value_counts()
        print(f"\n{'='*60}")
        print(f"✅ 뉴스 프레임 분석 완료! (GPT-5.4-mini)")
        print(f"{'='*60}")
        print(f"📊 총 {total_count:,}건 뉴스 분석 완료 (소요시간: {elapsed_time/60:.1f}분)")
        print(f"💰 총 비용: ${self.total_cost:.4f}")
        print(f"📁 결과 파일: {output_path}")
        print(f"\n📈 Semetko & Valkenburg 프레임 분포:")
        for frame, count in frame_dist.items():
            print(f"   • {frame}: {count:,}건 ({(count/total_count)*100:.1f}%)")
        print(f"\n📈 분석 성공률:")
        success_rate = (self.parsing_successes / total_count) * 100 if total_count > 0 else 0
        print(f"   • 성공: {self.parsing_successes}건 ({success_rate:.1f}%)")
        print(f"   • 실패: {self.parsing_failures}건 ({(self.parsing_failures/total_count)*100:.1f}%)")
        print(f"{'='*60}\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 5. 설정 및 실행
# ═══════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────
# 여기서 설정을 변경하세요
# ──────────────────────────────────────────────────────────────────────
MODEL = "gpt-5.4-mini"          # GPT-5.4-mini
API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")  # 환경변수 사용 권장. 키를 코드에 직접 적은 채 커밋하지 마세요.
INPUT_CSV_PATH  = "Input CSV file path here"
OUTPUT_CSV_PATH = "Output CSV file path here"

# GPT-5.4-mini 전용 설정
# reasoning_effort options: none | low | medium | high | xhigh
# ※ classification 태스크에는 'none' 권장
REASONING_EFFORT = "none"
TEXT_VERBOSITY   = "medium"     # low | medium | high
# temperature: reasoning_effort='none'일 때만 유효 (None이면 비활성화)
TEMPERATURE      = None
# ※ GPT-4.1 버전은 temperature=0.0 고정이었습니다. 동일한 재현성이 필요하면 0.0으로 설정하세요(reasoning_effort="none"일 때만 유효).
DEBUG_MODE       = False

# 비용 추정용 단가 (USD / 1M tokens). OpenAI 공식 가격표 기준 gpt-5.4-mini 입력 $0.75, 출력 $4.50 (2026-11 확인). 가격이 바뀌면 수정하세요.
PRICE_INPUT_PER_1M  = 0.75
PRICE_OUTPUT_PER_1M = 4.50
# ──────────────────────────────────────────────────────────────────────

async def main():
    try:
        df = pd.read_csv(INPUT_CSV_PATH, encoding='utf-8-sig')
        if 'title' not in df.columns or 'text' not in df.columns:
            print("❌ title, text 컬럼 필요")
            return

        output_dir  = os.path.dirname(OUTPUT_CSV_PATH) or '.'
        output_file = os.path.basename(OUTPUT_CSV_PATH)

        analyzer = NewsFrameAnalyzer(
            api_key=API_KEY,
            model=MODEL,
            output_dir=output_dir,
            reasoning_effort=REASONING_EFFORT,
            text_verbosity=TEXT_VERBOSITY,
            temperature=TEMPERATURE,
            debug=DEBUG_MODE
        )
        await analyzer.analyze_dataframe(df, output_file)

    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없음: {INPUT_CSV_PATH}")
    except Exception as e:
        print(f"❌ 오류: {str(e)}")
        import traceback; traceback.print_exc()

def run_analysis():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        pass
    asyncio.run(main())

if __name__ == "__main__":
    run_analysis()